# Inverse Perspective Mapping (IPM) for Lane Detection

**Learning Objectives:**
- Understand why Bird's Eye View (BEV) is useful for lane detection
- Implement the complete IPMTransform class
- Learn how to select calibration points for IPM
- Test and validate the transformation

**Prerequisites:** Completed Notebook 1 (Understanding Homography)

---

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import cv2
import sys
import os

# Set up matplotlib
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

print("Libraries imported successfully!")

## Import homography functions from Notebook 1

We'll use the functions we implemented in the first notebook.

In [ ]:
# Copy the key functions from Notebook 1

def to_homogeneous(points):
    """Convert Cartesian coordinates to homogeneous coordinates."""
    points = np.asarray(points, dtype=np.float32)
    if points.ndim == 1:
        points = points.reshape(1, -1)
    N = points.shape[0]
    ones = np.ones((N, 1))
    return np.hstack([points, ones])


def from_homogeneous(points_h):
    """Convert homogeneous coordinates to Cartesian coordinates."""
    points_h = np.asarray(points_h, dtype=np.float32)
    if points_h.ndim == 1:
        points_h = points_h.reshape(1, -1)
    w = points_h[:, 2:3]
    xy = points_h[:, :2]
    return xy / w


def is_collinear(points, tolerance=1e-6):
    """Check if points are collinear."""
    if len(points) < 3:
        return False
    p1, p2, p3 = points[:3]
    v1 = p2 - p1
    v2 = p3 - p1
    cross = v1[0] * v2[1] - v1[1] * v2[0]
    return abs(cross) < tolerance


def compute_homography(src_points, dst_points):
    """Compute homography using DLT algorithm (from Notebook 1)."""
    src = np.asarray(src_points, dtype=np.float32)
    dst = np.asarray(dst_points, dtype=np.float32)
    
    if src.shape != dst.shape:
        raise ValueError(f"Source and destination shapes must match: {src.shape} != {dst.shape}")
    if len(src.shape) != 2 or src.shape[1] != 2:
        raise ValueError(f"Points must be Nx2 array, got shape {src.shape}")
    
    n_points = src.shape[0]
    if n_points < 4:
        raise ValueError(f"Need at least 4 points, got {n_points}")
    
    if is_collinear(src):
        raise ValueError("Source points are collinear")
    if is_collinear(dst):
        raise ValueError("Destination points are collinear")
    
    # Build constraint matrix A
    A = np.zeros((2 * n_points, 9))
    for i in range(n_points):
        x, y = src[i]
        xp, yp = dst[i]
        A[2*i] = [-x, -y, -1, 0, 0, 0, xp*x, xp*y, xp]
        A[2*i + 1] = [0, 0, 0, -x, -y, -1, yp*x, yp*y, yp]
    
    # Solve using SVD
    U, S, Vt = np.linalg.svd(A)
    h = Vt[-1, :]
    H = h.reshape(3, 3)
    H = H / H[2, 2]
    
    return H


print("✓ Homography functions imported from Notebook 1")

---

## Section 1: The Perspective Projection Problem

### Why do we need Bird's Eye View (BEV)?

In front-view camera images:
- **Parallel lane lines converge** toward a vanishing point (perspective effect)
- **Distance is non-uniform**: Pixels near the camera represent smaller real-world distances than pixels far away
- **Lane detection is harder**: Curved appearance, varying width

In Bird's Eye View (BEV):
- **Parallel lanes stay parallel** (orthographic projection)
- **Uniform scale**: Each pixel represents the same real-world distance
- **Easier geometry**: Lane fitting, distance estimation, path planning

### The Flat Ground Assumption

IPM makes a critical assumption: **All points lie on a flat ground plane (Z=0)**.

This works well for:
- Flat highways and roads
- Parking lots
- Lane markings painted on the ground

This fails for:
- Hills and slopes (non-planar ground)
- Objects with height (pedestrians, vehicles)
- Curved roads (approximately works with piecewise approach)

In [ ]:
# Load a sample road image from KITTI dataset
image_path = '/home/vigoroth/mst_research/temporal 3D detection  BEV/data/KITTI.v2i.yolov12/test/images/000243_png.rf.0ec750f675d3bc21a5319bea8c2d2dbd.jpg'

if os.path.exists(image_path):
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    print(f"✓ Image loaded successfully")
    print(f"  Shape: {image.shape}")
    print(f"  Size: {image.shape[1]}x{image.shape[0]} (width × height)")
    
    plt.figure(figsize=(14, 8))
    plt.imshow(image)
    plt.title('Front-View Road Image from KITTI Dataset', fontsize=14, fontweight='bold')
    plt.axis('off')
    plt.show()
else:
    print(f"⚠ Image not found at: {image_path}")
    print("  Creating a synthetic road image for demonstration...")
    
    # Create synthetic road image
    height, width = 480, 640
    image = np.zeros((height, width, 3), dtype=np.uint8)
    image[:] = (100, 120, 140)  # Gray background
    
    # Draw converging lane lines (perspective)
    vp_x, vp_y = width//2, height//3  # Vanishing point
    # Left lane
    cv2.line(image, (100, height), (vp_x - 50, vp_y), (255, 255, 255), 3)
    # Right lane
    cv2.line(image, (width - 100, height), (vp_x + 50, vp_y), (255, 255, 255), 3)
    
    plt.figure(figsize=(14, 8))
    plt.imshow(image)
    plt.title('Synthetic Road Image (Demonstration)', fontsize=14, fontweight='bold')
    plt.axis('off')
    plt.show()
    
    print("✓ Synthetic image created")

---

## Section 2: Defining Source and Destination Regions

### Source Region (Front-View): Trapezoid

We select 4 points forming a **trapezoid** in the front-view image:
- **Top edge**: Narrower (farther from camera, perspective makes it smaller)
- **Bottom edge**: Wider (closer to camera)

Typical ratios (as fractions of image width/height):
```python
top_left = (0.4 * width, 0.6 * height)
top_right = (0.6 * width, 0.6 * height)
bottom_right = (0.9 * width, 0.95 * height)
bottom_left = (0.1 * width, 0.95 * height)
```

### Destination Region (BEV): Rectangle

In BEV, we want a **rectangle** (parallel sides):
```python
top_left = (0, 0)
top_right = (bev_width, 0)
bottom_right = (bev_width, bev_height)
bottom_left = (0, bev_height)
```

In [ ]:
# Define default ROI configuration
roi_config = {
    'top_left_ratio': (0.4, 0.6),
    'top_right_ratio': (0.6, 0.6),
    'bottom_right_ratio': (0.9, 0.95),
    'bottom_left_ratio': (0.1, 0.95),
    'bev_width': 640,
    'bev_height': 480
}

# Compute source points (trapezoid) from image
height, width = image.shape[:2]
src_points = np.array([
    [roi_config['top_left_ratio'][0] * width, roi_config['top_left_ratio'][1] * height],
    [roi_config['top_right_ratio'][0] * width, roi_config['top_right_ratio'][1] * height],
    [roi_config['bottom_right_ratio'][0] * width, roi_config['bottom_right_ratio'][1] * height],
    [roi_config['bottom_left_ratio'][0] * width, roi_config['bottom_left_ratio'][1] * height]
], dtype=np.float32)

# Compute destination points (rectangle) in BEV
dst_points = np.array([
    [0, 0],
    [roi_config['bev_width'], 0],
    [roi_config['bev_width'], roi_config['bev_height']],
    [0, roi_config['bev_height']]
], dtype=np.float32)

print("Source Points (Trapezoid in front-view):")
print(src_points)
print("\nDestination Points (Rectangle in BEV):")
print(dst_points)

# Visualize the ROI
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Front-view with trapezoid
ax1 = axes[0]
ax1.imshow(image)
# Draw trapezoid
for i in range(4):
    p1 = src_points[i]
    p2 = src_points[(i + 1) % 4]
    ax1.plot([p1[0], p2[0]], [p1[1], p2[1]], 'r-', linewidth=3)
ax1.scatter(src_points[:, 0], src_points[:, 1], s=200, c='red', zorder=3, edgecolor='white', linewidth=2)
for i, pt in enumerate(src_points):
    ax1.text(pt[0], pt[1]-20, f'P{i}', color='yellow', fontsize=12, fontweight='bold', ha='center')
ax1.set_title('Source ROI (Trapezoid)', fontsize=14, fontweight='bold')
ax1.axis('off')

# BEV rectangle
ax2 = axes[1]
bev_placeholder = np.ones((roi_config['bev_height'], roi_config['bev_width'], 3), dtype=np.uint8) * 200
ax2.imshow(bev_placeholder)
# Draw rectangle
for i in range(4):
    p1 = dst_points[i]
    p2 = dst_points[(i + 1) % 4]
    ax2.plot([p1[0], p2[0]], [p1[1], p2[1]], 'b-', linewidth=3)
ax2.scatter(dst_points[:, 0], dst_points[:, 1], s=200, c='blue', zorder=3, edgecolor='white', linewidth=2)
for i, pt in enumerate(dst_points):
    ax2.text(pt[0], pt[1]-20, f'P{i}\'', color='darkblue', fontsize=12, fontweight='bold', ha='center')
ax2.set_title('Destination ROI (Rectangle in BEV)', fontsize=14, fontweight='bold')
ax2.axis('off')

plt.tight_layout()
plt.show()

print("\n💡 The trapezoid (converging lines) in front-view maps to a rectangle (parallel lines) in BEV")

---

## Section 3: Compute Homography and Transform Image

Now we'll use `compute_homography()` to find the transformation matrix H and apply it using OpenCV's `warpPerspective()`.

In [ ]:
# Compute homography H from source to destination
H = compute_homography(src_points, dst_points)

print("Homography Matrix H:")
print(H)
print("\nThis matrix transforms front-view → BEV")

# Transform the image to BEV using cv2.warpPerspective
bev_image = cv2.warpPerspective(
    image, 
    H, 
    (roi_config['bev_width'], roi_config['bev_height']),
    flags=cv2.INTER_LINEAR,
    borderMode=cv2.BORDER_CONSTANT,
    borderValue=(0, 0, 0)
)

print("\n✓ BEV transformation complete!")
print(f"  BEV image shape: {bev_image.shape}")

# Display results side-by-side
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].imshow(image)
axes[0].set_title('Front-View (Original)', fontsize=14, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(bev_image)
axes[1].set_title('Bird\'s Eye View (BEV)', fontsize=14, fontweight='bold')
axes[1].axis('off')

plt.suptitle('IPM Transformation: Front-View → BEV', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Notice: Lane lines that were converging in front-view should now be more parallel in BEV!")

---

## Section 4: IPMTransform Class Implementation

Let's package everything into a reusable `IPMTransform` class with all required methods:

1. `__init__()` - Initialize with image shape and ROI config
2. `_compute_src_points()` - Compute trapezoid from ratios
3. `_compute_dst_points()` - Compute BEV rectangle
4. `transform_to_bev()` - Image transformation front → BEV
5. `transform_from_bev()` - Inverse image transformation BEV → front
6. `transform_points_to_bev()` - Point transformation (vectorized)
7. `transform_points_from_bev()` - Inverse point transformation

In [ ]:
class IPMTransform:
    """
    Inverse Perspective Mapping (IPM) transformation for Bird's Eye View (BEV).
    
    Transforms front-view images to overhead (BEV) representation using homography.
    Assumes flat ground plane (Z=0).
    """
    
    def __init__(self, image_shape, roi_config):
        """
        Initialize IPM transformation.
        
        Args:
            image_shape: tuple (height, width) of input image
            roi_config: dict with keys:
                - top_left_ratio: (x_ratio, y_ratio) for top-left corner
                - top_right_ratio: (x_ratio, y_ratio) for top-right corner
                - bottom_right_ratio: (x_ratio, y_ratio) for bottom-right corner
                - bottom_left_ratio: (x_ratio, y_ratio) for bottom-left corner
                - bev_width: output BEV width in pixels
                - bev_height: output BEV height in pixels
        
        Example:
            >>> config = {
            ...     'top_left_ratio': (0.4, 0.6),
            ...     'top_right_ratio': (0.6, 0.6),
            ...     'bottom_right_ratio': (0.9, 0.95),
            ...     'bottom_left_ratio': (0.1, 0.95),
            ...     'bev_width': 640,
            ...     'bev_height': 480
            ... }
            >>> ipm = IPMTransform((720, 1280), config)
        """
        self.image_height = image_shape[0]
        self.image_width = image_shape[1]
        self.roi_config = roi_config
        
        # Compute source and destination points
        self.src_points = self._compute_src_points(roi_config)
        self.dst_points = self._compute_dst_points(roi_config)
        
        # Compute forward and inverse homography matrices
        self.H = compute_homography(self.src_points, self.dst_points)
        self.H_inv = np.linalg.inv(self.H)
        
        self.bev_width = roi_config['bev_width']
        self.bev_height = roi_config['bev_height']
    
    def _compute_src_points(self, config):
        """
        Compute source trapezoid vertices from ratio configuration.
        
        Args:
            config: ROI configuration dict
        
        Returns:
            np.ndarray: (4, 2) array of corner points [top-left, top-right, bottom-right, bottom-left]
        """
        src_points = np.array([
            [config['top_left_ratio'][0] * self.image_width, 
             config['top_left_ratio'][1] * self.image_height],
            [config['top_right_ratio'][0] * self.image_width, 
             config['top_right_ratio'][1] * self.image_height],
            [config['bottom_right_ratio'][0] * self.image_width, 
             config['bottom_right_ratio'][1] * self.image_height],
            [config['bottom_left_ratio'][0] * self.image_width, 
             config['bottom_left_ratio'][1] * self.image_height]
        ], dtype=np.float32)
        
        return src_points
    
    def _compute_dst_points(self, config):
        """
        Compute destination rectangle vertices for BEV.
        
        Args:
            config: ROI configuration dict
        
        Returns:
            np.ndarray: (4, 2) array of corner points forming a rectangle
        """
        bev_w = config['bev_width']
        bev_h = config['bev_height']
        
        dst_points = np.array([
            [0, 0],                # top-left
            [bev_w, 0],            # top-right
            [bev_w, bev_h],        # bottom-right
            [0, bev_h]             # bottom-left
        ], dtype=np.float32)
        
        return dst_points
    
    def transform_to_bev(self, image):
        """
        Transform front-view image to Bird's Eye View.
        
        Args:
            image: np.ndarray of shape (H, W, 3) or (H, W)
        
        Returns:
            np.ndarray: BEV image of shape (bev_height, bev_width, 3) or (bev_height, bev_width)
        """
        bev_image = cv2.warpPerspective(
            image,
            self.H,
            (self.bev_width, self.bev_height),
            flags=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_CONSTANT,
            borderValue=(0, 0, 0) if len(image.shape) == 3 else 0
        )
        
        return bev_image
    
    def transform_from_bev(self, bev_image):
        """
        Transform BEV image back to front-view (inverse transformation).
        
        Args:
            bev_image: np.ndarray of shape (bev_height, bev_width, 3) or (bev_height, bev_width)
        
        Returns:
            np.ndarray: Front-view image of shape (image_height, image_width, 3) or (image_height, image_width)
        """
        front_image = cv2.warpPerspective(
            bev_image,
            self.H_inv,
            (self.image_width, self.image_height),
            flags=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_CONSTANT,
            borderValue=(0, 0, 0) if len(bev_image.shape) == 3 else 0
        )
        
        return front_image
    
    def transform_points_to_bev(self, points):
        """
        Transform points from front-view to BEV (vectorized).
        
        Args:
            points: np.ndarray of shape (N, 2) - points in front-view image coordinates
        
        Returns:
            np.ndarray of shape (N, 2) - points in BEV coordinates
        """
        points = np.asarray(points, dtype=np.float32)
        if points.ndim == 1:
            points = points.reshape(1, -1)
        
        # Convert to homogeneous
        points_h = to_homogeneous(points)
        
        # Apply homography
        transformed_h = (self.H @ points_h.T).T
        
        # Convert back to Cartesian
        transformed = from_homogeneous(transformed_h)
        
        return transformed
    
    def transform_points_from_bev(self, bev_points):
        """
        Transform points from BEV to front-view (inverse point transformation).
        
        Args:
            bev_points: np.ndarray of shape (N, 2) - points in BEV coordinates
        
        Returns:
            np.ndarray of shape (N, 2) - points in front-view image coordinates
        """
        bev_points = np.asarray(bev_points, dtype=np.float32)
        if bev_points.ndim == 1:
            bev_points = bev_points.reshape(1, -1)
        
        # Convert to homogeneous
        points_h = to_homogeneous(bev_points)
        
        # Apply inverse homography
        transformed_h = (self.H_inv @ points_h.T).T
        
        # Convert back to Cartesian
        transformed = from_homogeneous(transformed_h)
        
        return transformed


print("✓ IPMTransform class defined successfully!")
print("\nThe class has 7 methods:")
print("  1. __init__() - Initialize and compute H, H_inv")
print("  2. _compute_src_points() - Trapezoid from ratios")
print("  3. _compute_dst_points() - Rectangle for BEV")
print("  4. transform_to_bev() - Image: front → BEV")
print("  5. transform_from_bev() - Image: BEV → front")
print("  6. transform_points_to_bev() - Points: front → BEV")
print("  7. transform_points_from_bev() - Points: BEV → front")

---

## Section 5: Testing IPMTransform

Let's test all the methods of our IPMTransform class!

In [ ]:
# Create IPMTransform instance
print("Creating IPMTransform instance...")
ipm = IPMTransform(image.shape[:2], roi_config)
print("✓ IPMTransform created successfully!")

print(f"\nConfiguration:")
print(f"  Input image size: {ipm.image_width}x{ipm.image_height}")
print(f"  BEV output size: {ipm.bev_width}x{ipm.bev_height}")
print(f"\nSource points (trapezoid):\n{ipm.src_points}")
print(f"\nDestination points (rectangle):\n{ipm.dst_points}")
print(f"\nHomography H:\n{ipm.H}")

In [ ]:
# Test 1: Image transformation to BEV
print("Test 1: transform_to_bev()")
print("=" * 60)

bev = ipm.transform_to_bev(image)
print(f"✓ BEV transformation successful")
print(f"  Input shape: {image.shape}")
print(f"  BEV shape: {bev.shape}")

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].imshow(image)
axes[0].set_title('Front-View', fontsize=14, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(bev)
axes[1].set_title('BEV (transform_to_bev)', fontsize=14, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Test 2: Inverse transformation (BEV → front)
print("\nTest 2: transform_from_bev()")
print("=" * 60)

front_reconstructed = ipm.transform_from_bev(bev)
print(f"✓ Inverse transformation successful")
print(f"  BEV shape: {bev.shape}")
print(f"  Reconstructed front-view shape: {front_reconstructed.shape}")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(image)
axes[0].set_title('Original Front-View', fontsize=13, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(bev)
axes[1].set_title('BEV', fontsize=13, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(front_reconstructed)
axes[2].set_title('Reconstructed (BEV → Front)', fontsize=13, fontweight='bold')
axes[2].axis('off')

plt.suptitle('Round-Trip: Front → BEV → Front', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 The round-trip transformation shows information within the ROI is preserved!")

In [ ]:
# Test 3: Point transformation
print("\nTest 3: transform_points_to_bev() and transform_points_from_bev()")
print("=" * 60)

# Define test points in front-view
test_points = np.array([
    [width * 0.3, height * 0.7],
    [width * 0.5, height * 0.7],
    [width * 0.7, height * 0.7],
    [width * 0.5, height * 0.9]
], dtype=np.float32)

# Transform to BEV
bev_points = ipm.transform_points_to_bev(test_points)
print(f"✓ Points transformed to BEV")
print(f"  Front-view points:\n{test_points}")
print(f"  BEV points:\n{bev_points}")

# Transform back to front-view (round-trip test)
front_points_reconstructed = ipm.transform_points_from_bev(bev_points)
print(f"\n✓ Points transformed back to front-view")
print(f"  Reconstructed points:\n{front_points_reconstructed}")

# Compute error
error = np.linalg.norm(test_points - front_points_reconstructed, axis=1)
print(f"\nRound-trip error (pixels):")
for i, err in enumerate(error):
    print(f"  Point {i}: {err:.6f} pixels")
print(f"  Mean error: {error.mean():.6f} pixels")
print(f"  Max error: {error.max():.6f} pixels")

if error.max() < 1e-3:
    print("\n✓ Round-trip transformation is accurate (error < 0.001 pixels)!")
else:
    print("\n⚠ Small numerical errors detected (expected due to floating-point precision)")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Front-view
axes[0].imshow(image)
axes[0].scatter(test_points[:, 0], test_points[:, 1], s=200, c='red', marker='o', 
               edgecolor='white', linewidth=2, label='Test Points', zorder=3)
for i, pt in enumerate(test_points):
    axes[0].text(pt[0], pt[1]-25, f'P{i}', color='yellow', fontsize=12, 
                fontweight='bold', ha='center')
axes[0].set_title('Front-View with Test Points', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].axis('off')

# BEV
axes[1].imshow(bev)
axes[1].scatter(bev_points[:, 0], bev_points[:, 1], s=200, c='lime', marker='^', 
               edgecolor='white', linewidth=2, label='Transformed Points', zorder=3)
for i, pt in enumerate(bev_points):
    axes[1].text(pt[0], pt[1]-25, f'P{i}\'', color='yellow', fontsize=12, 
                fontweight='bold', ha='center')
axes[1].set_title('BEV with Transformed Points', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].axis('off')

plt.suptitle('Point Transformation: Front-View → BEV', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Summary: IPMTransform Class

### What We've Implemented:

1. **Complete IPMTransform Class** with 7 methods for image and point transformations
2. **Configurable ROI** using ratio-based trapezoid definition
3. **Precomputed transformations** (H and H_inv computed during `__init__`)
4. **Vectorized point operations** for efficiency

### Key Features:

- **Forward transformation**: `transform_to_bev()` - Convert front-view images to BEV
- **Inverse transformation**: `transform_from_bev()` - Convert BEV back to front-view
- **Point transformations**: Work with both images and individual points
- **Round-trip accuracy**: Transformations are reversible with minimal numerical error

### When IPM Works:
- Flat roads and highways ✅
- Parking lots ✅
- Lane markings on ground ✅

### When IPM Fails:
- Hills and slopes ❌ (violates Z=0 assumption)
- Objects with height (pedestrians, cars) ❌ (appears distorted)
- Highly curved roads ⚠️ (works with piecewise approach)

### Next Steps:

In **Notebook 3**, we'll explore failure modes in detail and understand when to use IPM vs. other approaches like deep learning-based BEV methods (LSS, BEVFormer).

After completing the notebooks, we'll extract this code to `src/ipm.py` and `src/homography.py` for production use!